# LAB 07 — Experiments (training)

Chạy **Run All** để huấn luyện lại từ đầu. Notebook không hiển thị bảng/hình có sẵn; chỉ in tiến độ training và lưu CSV, checkpoint, JSON, PNG vào `training_output/`. Kết quả cũ không được nhúng vào notebook.

Cần notebook, `transformer_block.py`, `positional_encoding.py`, dữ liệu IMDb CSV và các thư viện trong requirements.txt. Chọn runtime GPU và cài PyTorch hỗ trợ CUDA để dùng NVIDIA GPU. `DEVICE='auto'` tự chọn CUDA hoặc CPU; `DEVICE='cuda'` báo lỗi nếu không dùng được CUDA. Đổi `EPOCHS`, `DATASET_PATH`, `OUTPUT_DIR` ở cell cấu hình. Dữ liệu 4.000 mẫu; train/validation/test=2.800/400/800; seed 42.

In [1]:
from pathlib import Path
import sys
import torch

# Sửa đường dẫn này khi chạy trên Colab/Kaggle hoặc máy khác.
DATASET_PATH = Path('datasets/imdb/IMDB Dataset.csv').resolve()
if not DATASET_PATH.exists():
    DATASET_PATH = Path('../datasets/imdb/IMDB Dataset.csv').resolve()
OUTPUT_DIR = Path('training_output').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hai module này đặt cùng notebook, hoặc trong lab07/.
MODULE_DIR = Path.cwd() if (Path.cwd() / 'transformer_block.py').exists() else Path.cwd() / 'lab07'
sys.path.insert(0, str(MODULE_DIR))
EPOCHS = 4
DEVICE = 'auto'  # auto: CUDA nếu có, ngược lại CPU; đặt 'cuda' để bắt buộc GPU.
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Sửa DATASET_PATH cho đúng: {DATASET_PATH}')


## Hàm dùng chung
Tokenizer, dataset, classifier, train/evaluate và lưu kết quả. Model và batch được chuyển sang cùng device; checkpoint lưu tensor CPU để có thể nạp trên máy khác.

In [ ]:
"""Reproducible LAB 07 experiments. Run from any working directory."""
from pathlib import Path
import argparse, collections, json, math, platform, random, re, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from transformer_block import TransformerBlock
from positional_encoding import PositionalEncoding

ROOT = OUTPUT_DIR
SEED = 42
DATA_PATH = DATASET_PATH

def select_device(device="auto"):
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    selected = torch.device(device)
    if selected.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable. Install a CUDA-enabled PyTorch build and use a compatible NVIDIA GPU, or choose cpu/auto.")
    return selected

def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def seed():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def tokenize(text):
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', re.sub(r'<br\s*/?>', ' ', text.lower())).split()

class Classifier(nn.Module):
    def __init__(self, vocab_size, heads=4, pooling='mean', kind='Transformer', positions=True):
        super().__init__()
        self.kind, self.pooling, self.positions = kind, pooling, positions
        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.pe = PositionalEncoding(128)
        if kind == 'LSTM':
            self.encoder = nn.LSTM(128, 128, batch_first=True)
        else:
            self.encoder = TransformerBlock(128, heads, 256)
        self.classifier = nn.Linear(128, 2)

    def forward(self, ids, return_attention=False):
        mask = ids.eq(0)
        x = self.embedding(ids)
        if self.kind == 'LSTM':
            packed = nn.utils.rnn.pack_padded_sequence(x, (~mask).sum(1).cpu(), batch_first=True, enforce_sorted=False)
            _, (hidden, _) = self.encoder(packed)
            pooled, attention = hidden[-1], None
        else:
            if self.positions: x = self.pe(x)
            x, attention = self.encoder(x, mask, True)
            pooled = x[:, 0] if self.pooling == 'cls' else (x * (~mask).unsqueeze(-1)).sum(1) / (~mask).sum(1, keepdim=True)
        logits = self.classifier(pooled)
        return (logits, attention) if return_attention else logits

def prepare():
    source = DATA_PATH
    raw = pd.read_csv(source)
    # Same selection/order as the recoverable LAB 06 notebook; LAB 05 is absent.
    selected = pd.concat([raw[raw.sentiment.eq('positive')].head(2000), raw[raw.sentiment.eq('negative')].head(2000)])
    selected = selected.sample(frac=1, random_state=SEED).reset_index().rename(columns={'index':'source_row'})
    selected['split'] = ['train'] * 2800 + ['validation'] * 400 + ['test'] * 800
    # Keep held-out sets intact; replace training reviews duplicated in them.
    heldout = set(selected.iloc[2800:].review)
    used = set(selected.review)
    replacements = []
    for i in range(2800):
        if selected.at[i, 'review'] in heldout:
            label = selected.at[i, 'sentiment']
            candidate = raw[(raw.sentiment == label) & ~raw.review.isin(used)].iloc[0]
            replacements.append({'training_position': i, 'old_source_row': int(selected.at[i, 'source_row']), 'new_source_row': int(candidate.name), 'sentiment': label})
            selected.at[i, 'source_row'] = int(candidate.name)
            selected.at[i, 'review'] = candidate.review
            used.add(candidate.review)
    train_reviews = set(selected.iloc[:2800].review)
    assert not train_reviews & heldout
    assert not set(selected.iloc[2800:3200].review) & set(selected.iloc[3200:].review)
    (ROOT/'data_audit.json').write_text(json.dumps({'replacements': replacements, 'train_heldout_exact_overlap_after': 0, 'validation_test_exact_overlap': 0}, indent=2), encoding='utf-8')
    selected[['source_row','sentiment','split']].to_csv(ROOT/'split_manifest.csv',index=False)
    texts = [tokenize(t) for t in selected.review]
    counts = collections.Counter(w for seq in texts[:2800] for w in seq)
    vocab = {'<PAD>':0,'<UNK>':1,'<CLS>':2}
    vocab.update({word:i+3 for i,(word,_) in enumerate(counts.most_common(7500))})
    (ROOT/'vocab.json').write_text(json.dumps(vocab),encoding='utf-8')
    return selected,texts,vocab

def encode(texts,vocab,length,pooling='mean'):
    encoded=[]
    for tokens in texts:
        ids=[vocab.get(w,1) for w in tokens][:length]
        if pooling=='cls': ids=[2]+ids # Same review-token budget; one extra CLS position.
        if not ids: ids=[1]
        width=length+(pooling=='cls')
        encoded.append(ids+[0]*(width-len(ids)))
    return torch.tensor(encoded)

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); total=0.; predictions=[]; targets=[]; probabilities=[]
    device = next(model.parameters()).device
    for x,y in loader:
        x, y = x.to(device), y.to(device)
        logits=model(x); total+=nn.functional.cross_entropy(logits,y,reduction='sum').item()
        predictions.extend(logits.argmax(1).tolist());targets.extend(y.tolist());probabilities.extend(logits.softmax(1)[:,1].tolist())
    p,r,f,_=precision_recall_fscore_support(targets,predictions,average='macro',zero_division=0)
    return dict(loss=total/len(targets),accuracy=accuracy_score(targets,predictions),precision=p,recall=r,f1=f),predictions,probabilities

def loaders(ids,labels):
    gen=torch.Generator().manual_seed(SEED)
    return [DataLoader(TensorDataset(ids[a:b],labels[a:b]),batch_size=32,shuffle=(a==0),generator=gen if a==0 else None) for a,b in [(0,2800),(2800,3200),(3200,4000)]]

def permutation_experiment():
    seed(); block=TransformerBlock(4,2,8,dropout=0).eval(); pe=PositionalEncoding(4)
    embedding=torch.tensor([[1.,0.,0.,1.],[0.,1.,1.,0.],[1.,1.,0.,0.]])[None]
    permutation=torch.tensor([2,1,0]); reversed_x=embedding[:,permutation]
    report={}
    with torch.no_grad():
        for positions in [False,True]:
            a=block(pe(embedding) if positions else embedding)
            b=block(pe(reversed_x) if positions else reversed_x)
            report['with_pe' if positions else 'without_pe']={
                'aligned_token_max_abs_difference':(a[:,permutation]-b).abs().max().item(),
                'mean_pool_max_abs_difference':(a.mean(1)-b.mean(1)).abs().max().item(),
                's1_token_representations':a.squeeze(0).tolist(), 's2_token_representations':b.squeeze(0).tolist()}
    # Both pairs are permutations of three distinct tokens; assign the same demonstration vectors.
    report['pairs']=[['I love NLP','NLP love I'],['dog bites man','man bites dog']]
    report['note']='The same fixed three-vector embedding is used for each pair; no training, eval mode, dropout=0.'
    (ROOT/'permutation_results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    fig,axes=plt.subplots(1,3,figsize=(12,3))
    for ax,matrix,title in zip(axes,[embedding[0],pe.pe[0,:3],pe(embedding)[0]],['Token embeddings','Sinusoidal PE','Embedding + PE']):
        im=ax.imshow(matrix.numpy(),aspect='auto',cmap='coolwarm');ax.set_title(title);ax.set_yticks(range(3),['I','love','NLP']);ax.set_xlabel('Feature');fig.colorbar(im,ax=ax)
    fig.tight_layout();fig.savefig(ROOT/'positional_encoding.png',dpi=180);plt.close(fig)
    return report

def checks():
    seed(); model=TransformerBlock(128,4,256,dropout=0).eval()
    x=torch.randn(8,100,128)
    with torch.no_grad():
        out,att=model(x,return_attention=True)
        assert out.shape==(8,100,128) and att.shape==(8,4,100,100)
        assert torch.allclose(att.sum(-1),torch.ones(8,4,100),atol=1e-6)
        short=x[:1,:3]; padded=torch.cat([short,torch.randn(1,2,128)],dim=1)
        mask=torch.tensor([[False,False,False,True,True]])
        a=model(short); b,w=model(padded,mask,True)
        assert torch.allclose(a,b[:,:3],atol=2e-6)
        assert w[:,:,:,-2:].abs().max()==0 and b[:,3:].abs().max()==0
    # Independent reference verifies projections, scaling, head merge and mask.
    reference=nn.MultiheadAttention(128,4,dropout=0,batch_first=True)
    mha=model.attention
    with torch.no_grad():
        reference.in_proj_weight.copy_(torch.cat([mha.q.weight,mha.k.weight,mha.v.weight]))
        reference.in_proj_bias.copy_(torch.cat([mha.q.bias,mha.k.bias,mha.v.bias]))
        reference.out_proj.load_state_dict(mha.out.state_dict())
        expected,_=reference(padded,padded,padded,key_padding_mask=mask)
        actual=mha(padded,mask)
        assert torch.allclose(expected[:,:3],actual[:,:3],atol=2e-6)
    model.train(); model(x).square().mean().backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    result={'input':[8,100,128],'q_k_v':[8,4,100,32],'scores':[8,4,100,100], 'head_output':[8,4,100,32], 'concat':[8,100,128], 'ffn_hidden':[8,100,256], 'output':list(out.shape),'checks':'shape, softmax, padding invariance, reference attention, finite gradients passed'}
    (ROOT/'shape_checks.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    return result

def run(epochs=4, device="auto", experiment_names=None, finalize=True):
    if epochs < 1:
        raise ValueError("epochs must be positive")
    device = select_device(device)
    print(f"Training device: {device}", flush=True)
    torch.set_num_threads(4); seed();checks(); permutation_experiment()
    frame,texts,vocab=prepare();labels=torch.tensor(frame.sentiment.eq('positive').astype(int).to_numpy())
    config={'seed':SEED,'epochs':epochs,'batch_size':32,'learning_rate':0.001,'optimizer':'Adam','d_model':128,'d_ff':256,'dropout':0.1,'threads':4,'train':2800,'validation':400,'test':800,'device':str(device),'gpu':torch.cuda.get_device_name(device) if device.type == 'cuda' else None,'torch':torch.__version__,'python':platform.python_version(),'selection':'first 2000 per class, shuffle seed 42','checkpoint':'minimum validation loss','metrics':'macro precision/recall/F1','dataset':'local IMDB Dataset.csv; random review split, not official IMDb split'}
    (ROOT/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    # Freeze predictions before any trained-model experiments.
    # Predictions are a separate written deliverable; training does not depend on it.
    experiments=[('heads_1','Transformer',1,64,'mean',True),('heads_2','Transformer',2,64,'mean',True),('baseline','Transformer',4,64,'mean',True),('heads_8','Transformer',8,64,'mean',True),('lstm','LSTM',4,64,'mean',False),('length_32','Transformer',4,32,'mean',True),('length_128','Transformer',4,128,'mean',True),('length_256','Transformer',4,256,'mean',True),('pool_cls','Transformer',4,64,'cls',True),('no_pe','Transformer',4,64,'mean',False)]
    if experiment_names is not None:
        unknown = set(experiment_names) - {item[0] for item in experiments}
        if unknown:
            raise ValueError(f"Unknown experiments: {unknown}")
        experiments = [item for item in experiments if item[0] in experiment_names]
    rows=[];histories={};baseline=None
    if experiment_names is not None:
        if (ROOT/'results.csv').exists():
            rows = pd.read_csv(ROOT/'results.csv').to_dict('records')
        if (ROOT/'history.json').exists():
            histories = json.loads((ROOT/'history.json').read_text())
    for name,kind,heads,length,pooling,positions in experiments:
        seed(); ids=encode(texts,vocab,length,pooling);train,val,test=loaders(ids,labels)
        model=Classifier(len(vocab),heads,pooling,kind,positions).to(device)
        optimizer=torch.optim.Adam(model.parameters(),lr=.001)
        history=[]; best_loss=float('inf');best_state=None; synchronize(device); started=time.perf_counter()
        for epoch in range(epochs):
            model.train(); loss_sum=0.;synchronize(device);epoch_start=time.perf_counter()
            for x,y in train:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad(); loss=nn.functional.cross_entropy(model(x),y);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step();loss_sum+=loss.item()*len(y)
            synchronize(device)
            epoch_training=time.perf_counter()-epoch_start
            metrics,_,_=evaluate(model,val)
            history.append({'epoch':epoch+1,'train_loss':loss_sum/2800,'validation_loss':metrics['loss'],'validation_f1':metrics['f1'],'training_seconds':epoch_training})
            if metrics['loss']<best_loss:
                best_loss=metrics['loss'];best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};best_epoch=epoch+1
            print(f'{name}: epoch {epoch+1}/{epochs} train={loss_sum/2800:.4f} val={metrics["loss"]:.4f}',flush=True)
        synchronize(device)
        elapsed=time.perf_counter()-started
        model.load_state_dict(best_state);metrics,preds,probs=evaluate(model,test)
        retained=sum(min(len(t),length) for t in texts[3200:])/sum(len(t) for t in texts[3200:])
        width=length+(pooling=='cls')
        row=dict(experiment=name,model=kind,heads=heads if kind!='LSTM' else 0,d_k=128//heads if kind!='LSTM' else 0,max_length=length,pooling=pooling if kind!='LSTM' else 'last_hidden',positions=positions,parameters=sum(p.numel() for p in model.parameters()),training_seconds=sum(h['training_seconds'] for h in history),fit_wall_seconds=elapsed,best_epoch=best_epoch,tokens_retained=retained,attention_mib=32*heads*width*width*4/2**20 if kind!='LSTM' else 0,**metrics)
        rows = [old for old in rows if old['experiment'] != name]
        rows.append(row);histories[name]=history
        pd.DataFrame(rows).to_csv(ROOT/'results.csv',index=False)
        (ROOT/'history.json').write_text(json.dumps(histories,indent=2),encoding='utf-8')
        print(f'Finished {name}: F1={metrics["f1"]:.4f} training={row["training_seconds"]:.1f}s',flush=True)
        if name=='baseline':
            baseline=model
            torch.save({'state_dict':best_state,'vocab':vocab,'max_length':64,'heads':4,'d_model':128,'d_ff':256},ROOT/'transformer_model.pt')
            predictions=frame.iloc[3200:].copy();predictions['target']=labels[3200:].tolist();predictions['prediction']=preds;predictions['positive_probability']=probs
            predictions.to_csv(ROOT/'test_predictions.csv',index=False)
    if finalize:
        finalize_experiments(device)
    return pd.DataFrame(rows)
    print('All experiments completed.',flush=True)

def finalize_experiments(device="auto"):
    device = select_device(device)
    frame, texts, vocab = prepare()
    labels = torch.tensor(frame.sentiment.eq('positive').astype(int).to_numpy())
    checkpoint = torch.load(ROOT/'transformer_model.pt', map_location='cpu', weights_only=True)
    model = Classifier(len(vocab), heads=4).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    rows = pd.read_csv(ROOT/'results.csv').to_dict('records')
    histories = json.loads((ROOT/'history.json').read_text())
    make_plots(rows,histories,model,texts,vocab,labels)
    application(model,vocab)

def make_plots(rows,histories,model,texts,vocab,labels):
    fig,axes=plt.subplots(2,5,figsize=(19,7))
    for ax,(name,history) in zip(axes.flat,histories.items()):
        ax.plot([h['epoch'] for h in history],[h['train_loss'] for h in history],label='Train')
        ax.plot([h['epoch'] for h in history],[h['validation_loss'] for h in history],label='Validation');ax.set_title(name);ax.set_xlabel('Epoch');ax.set_ylabel('Cross entropy');ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(ROOT/'learning_curves.png',dpi=160);plt.close(fig)
    pred=pd.read_csv(ROOT/'test_predictions.csv');cm=confusion_matrix(pred.target,pred.prediction,labels=[0,1])
    fig,ax=plt.subplots(figsize=(5,4));ax.imshow(cm,cmap='Blues')
    for i in range(2):
        for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Negative','Positive'],yticklabels=['Negative','Positive'],xlabel='Predicted',ylabel='True',title='Transformer, 4 heads, length 64');fig.tight_layout();fig.savefig(ROOT/'confusion_matrix.png',dpi=180);plt.close(fig)
    sentence='the movie was not good but the acting was excellent'
    tokens=tokenize(sentence);ids=encode([tokens],vocab,64).to(next(model.parameters()).device)
    model.eval()
    with torch.no_grad(): _,att=model(ids,True)
    matrices=att[0,:,:len(tokens),:len(tokens)].cpu().numpy()
    fig,axes=plt.subplots(1,4,figsize=(18,5));stats=[]
    for h,ax in enumerate(axes):
        ax.imshow(matrices[h],vmin=0,vmax=matrices.max(),cmap='viridis');ax.set_title(f'Head {h+1}');ax.set_xticks(range(len(tokens)),tokens,rotation=90);ax.set_yticks(range(len(tokens)),tokens);ax.set_xlabel('Key');ax.set_ylabel('Query')
        entropy=-(matrices[h]*np.log(matrices[h]+1e-12)).sum(-1).mean()
        stats.append({'head':h+1,'mean_entropy':float(entropy),'mean_max_weight':float(matrices[h].max(-1).mean())})
    fig.tight_layout();fig.savefig(ROOT/'attention_heads.png',dpi=180);plt.close(fig)
    differences=[{'heads':[i+1,j+1],'mean_absolute_difference':float(np.abs(matrices[i]-matrices[j]).mean())} for i in range(4) for j in range(i+1,4)]
    (ROOT/'head_diversity.json').write_text(json.dumps({'sentence':sentence,'stats':stats,'pairs':differences},indent=2),encoding='utf-8')
    # Store attention observations for six genuine held-out examples.
    chosen=pd.concat([pred[pred.target.eq(pred.prediction)].head(3),pred[pred.target.ne(pred.prediction)].head(3)])
    observations=[]
    for _,r in chosen.iterrows():
        tok=tokenize(r.review); ids=encode([tok],vocab,64).to(next(model.parameters()).device)
        with torch.no_grad():_,a=model(ids,True)
        n=min(64,len(tok)); importance=a[0,:,:n,:n].mean((0,1)).cpu().numpy();top=importance.argsort()[-5:][::-1]
        observations.append({'source_row':int(r.source_row),'target':int(r.target),'prediction':int(r.prediction),'positive_probability':float(r.positive_probability),'tokens':len(tok),'visible_text':' '.join(tok[:64]),'full_review':r.review,'oov_fraction':sum(w not in vocab for w in tok[:64])/n,'attention_top_keys':[{'position':int(i),'token':tok[i],'mean_received_attention':float(importance[i])} for i in top]})
    (ROOT/'error_examples.json').write_text(json.dumps(observations,ensure_ascii=False,indent=2),encoding='utf-8')

def application(model,vocab):
    cases=[('short','I really enjoyed this movie.'),('long','The movie follows a family through many years of changing relationships. '*12+'The ending is deeply moving and I loved the film.'),('negation','This movie is not good. I did not enjoy it.'),('contrast','The acting was excellent, but the story was boring and disappointing.'),('ambiguous','Some moments worked for me, while others left me unsure how I felt.')]
    records=[];model.eval()
    for category,text in cases:
        tokens=tokenize(text)
        with torch.no_grad():p=model(encode([tokens],vocab,64).to(next(model.parameters()).device)).softmax(-1)[0].tolist()
        records.append({'category':category,'text':text,'tokens':len(tokens),'truncated':len(tokens)>64,'prediction':'Positive' if p[1]>=p[0] else 'Negative','positive_probability':p[1]})
    pd.DataFrame(records).to_csv(ROOT/'application_predictions.csv',index=False)



## Experiment 1 — Positional encoding và permutation

In [ ]:
checks()
_ = permutation_experiment()

## Experiment 2 — Number of attention heads


In [4]:
_ = run(epochs=EPOCHS, device=DEVICE, experiment_names=['heads_1','heads_2','baseline','heads_8'], finalize=False)

Training device: cuda
heads_1: epoch 1/4 train=0.6685 val=0.6120
heads_1: epoch 2/4 train=0.5997 val=0.5526
heads_1: epoch 3/4 train=0.5166 val=0.5674
heads_1: epoch 4/4 train=0.4254 val=0.5346
Finished heads_1: F1=0.7086 training=1.5s
heads_2: epoch 1/4 train=0.6697 val=0.6010
heads_2: epoch 2/4 train=0.5941 val=0.5552
heads_2: epoch 3/4 train=0.5248 val=0.5265
heads_2: epoch 4/4 train=0.4345 val=0.5322
Finished heads_2: F1=0.6749 training=1.3s
baseline: epoch 1/4 train=0.6741 val=0.6046
baseline: epoch 2/4 train=0.5782 val=0.5690
baseline: epoch 3/4 train=0.4903 val=0.5651
baseline: epoch 4/4 train=0.3774 val=0.5692
Finished baseline: F1=0.6689 training=1.4s
heads_8: epoch 1/4 train=0.6774 val=0.6283
heads_8: epoch 2/4 train=0.5794 val=0.5477
heads_8: epoch 3/4 train=0.4665 val=0.5084
heads_8: epoch 4/4 train=0.3548 val=0.5363
Finished heads_8: F1=0.6909 training=1.3s


## Experiment 3 — Attention head diversity


In [ ]:
finalize_experiments(device=DEVICE)

## Experiment 4 — Transformer vs LSTM


In [6]:
_ = run(epochs=EPOCHS, device=DEVICE, experiment_names=['lstm'], finalize=False)

Training device: cuda
lstm: epoch 1/4 train=0.6935 val=0.6903
lstm: epoch 2/4 train=0.6465 val=0.6975
lstm: epoch 3/4 train=0.5497 val=0.7138
lstm: epoch 4/4 train=0.4032 val=0.7958
Finished lstm: F1=0.5230 training=2.1s


## Experiment 5 — Sequence length


In [7]:
_ = run(epochs=EPOCHS, device=DEVICE, experiment_names=['length_32','length_128','length_256'], finalize=False)

Training device: cuda
length_32: epoch 1/4 train=0.6854 val=0.6236
length_32: epoch 2/4 train=0.6233 val=0.6507
length_32: epoch 3/4 train=0.5406 val=0.5974
length_32: epoch 4/4 train=0.4513 val=0.6244
Finished length_32: F1=0.6359 training=1.3s
length_128: epoch 1/4 train=0.6618 val=0.6287
length_128: epoch 2/4 train=0.5755 val=0.5435
length_128: epoch 3/4 train=0.4759 val=0.5413
length_128: epoch 4/4 train=0.3896 val=0.5006
Finished length_128: F1=0.7315 training=1.4s
length_256: epoch 1/4 train=0.6553 val=0.5794
length_256: epoch 2/4 train=0.5632 val=0.5019
length_256: epoch 3/4 train=0.4296 val=0.5013
length_256: epoch 4/4 train=0.3271 val=0.4473
Finished length_256: F1=0.7825 training=2.4s


## Experiment 6 — Pooling strategy


In [8]:
_ = run(epochs=EPOCHS, device=DEVICE, experiment_names=['pool_cls','no_pe'], finalize=False)
finalize_experiments(device=DEVICE)

Training device: cuda
pool_cls: epoch 1/4 train=0.6833 val=0.6296
pool_cls: epoch 2/4 train=0.6180 val=0.5899
pool_cls: epoch 3/4 train=0.5696 val=0.5834
pool_cls: epoch 4/4 train=0.5037 val=0.6194
Finished pool_cls: F1=0.6524 training=1.2s
no_pe: epoch 1/4 train=0.6706 val=0.6363
no_pe: epoch 2/4 train=0.5671 val=0.5533
no_pe: epoch 3/4 train=0.4604 val=0.5206
no_pe: epoch 4/4 train=0.3613 val=0.5505
Finished no_pe: F1=0.7057 training=1.3s
